# Practical Gassmann Fluid Substitution in Sand/Shale Sequences

**Reference:** Simm, R. (2007). *Practical rock physics*. First Break, Vol 25, December 2007.

**Well:** WELL-2, Glitne Field, Norway (Heimdal Formation — oil field)
**Data:** Vp (km/s), Vs (km/s), RHOB (g/cc), GR (GAPI), NPHI

---

## Workflow Overview — Simm's 6-Step Adaptive Approach

| Step | Description |
|------|-------------|
| 1 | Load well logs (Vp, Vs, density, GR) and assign facies labels |
| 2 | Compute Vsh from GR; effective porosity from density log |
| 3 | Mineral modulus K₀ via Voigt-Reuss-Hill (VRH) mixing (quartz + shale) |
| 4 | Gassmann inversion → dry-rock parameters (Kd/K₀, Kφ/K₀) |
| 5 | QC on Kd/K₀ vs φ crossplot; fit adaptive polynomial dry-rock trend |
| 6 | Forward Gassmann substitution: brine → oil  AND  brine → gas |

The key innovation in Simm's approach is **step 5**: instead of using the raw inverted
dry-frame modulus for every sample, a polynomial trend is fitted to the cleanest
reference sands and applied **selectively** to intermediate (silty) lithologies.
This prevents noisy inversion results from propagating into the forward substitution.

## 1. Imports

Standard scientific Python stack. `%matplotlib inline` renders figures directly
in the notebook rather than writing them to disk. The `glob` module is used to
discover per-facies data files by wildcard pattern.

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 2. Mineral and Fluid Constants

These constants define the **end-member physical properties** for all rock physics
calculations that follow:

- **Mineral properties** (quartz, clay/shale): elastic moduli (bulk K, shear G in GPa)
  and grain density (g/cc) from Simm (2007) and `info_params.txt`.
- **Fluid properties** (brine, gas): bulk modulus and density at reservoir conditions.
  Oil properties are computed from Batzle-Wang (1992) in the next section.
- **Reservoir conditions**: temperature, pressure, API gravity, and GOR feed the
  live-oil calculation. Depth markers (Top Heimdal, OWC) annotate every log plot.

These values anchor the VRH mineral mixing and the Gassmann substitution
throughout the entire workflow.

In [ ]:
# ── Quartz (info_params.txt) ──────────────────────────────────────────────────
K_QTZ   = 36.8;  G_QTZ   = 44.0;  RHO_QTZ  = 2.65   # GPa, g/cc

# ── Clay / Shale (info_params.txt) ────────────────────────────────────────────
K_SH    = 15.0;  G_SH    =  5.0;  RHO_SH   = 2.72

# ── Brine (info_params.txt) ───────────────────────────────────────────────────
K_BR    = 2.80;  RHO_BR  = 1.09   # GPa, g/cc

# ── Dry gas at reservoir conditions ───────────────────────────────────────────
K_GAS   = 0.02;  RHO_GAS = 0.12   # GPa, g/cc

# ── Oil: density given directly; K_OIL computed via Batzle-Wang below ─────────
RHO_OIL = 0.78                     # g/cc

# ── Reservoir conditions ──────────────────────────────────────────────────────
RESERVOIR_TEMP   = 77.2    # °C
RESERVOIR_P      = 20.0    # MPa effective pressure
OIL_API          = 32
OIL_GOR          = 64      # sm³/sm³  (metric solution GOR)
OWC_DEPTH        = 2183    # m  (oil-water contact)
TOP_HEIMDAL      = 2153    # m  (top of Heimdal reservoir)

print("Constants loaded.")
print(f"  Quartz:  K={K_QTZ} GPa, G={G_QTZ} GPa, ρ={RHO_QTZ} g/cc")
print(f"  Shale:   K={K_SH} GPa,  G={G_SH} GPa,  ρ={RHO_SH} g/cc")
print(f"  Brine:   K={K_BR} GPa,  ρ={RHO_BR} g/cc")
print(f"  Gas:     K={K_GAS} GPa, ρ={RHO_GAS} g/cc")

## 3. Data Loading

The well data are stored as whitespace-delimited ASCII text files — one master log
file (`well_2.txt`) and one file per facies population (`well2_clnSand.txt`, etc.).

**`_parse_txt`** — low-level parser that skips comment/header lines, returning a
DataFrame with columns `[depth, Vp, Vs, rho, GR, nphi]`.

**`load_well`** — reads the full well log for WELL-2.

**`load_facies`** — reads each per-facies file and collects the depths belonging
to that lithology class. Six facies are recognised:
*Clean Sand, Cemented Sand, Silty Sand 1, Silty Sand 2, Silty Shale, Shale.*

**`assign_facies`** — depth-matches the facies populations back onto the main well
DataFrame, adding a `facies` column used for colour-coding every subsequent plot.

In [ ]:
def _parse_txt(filepath):
    """Parse a well-data text file (handles both header styles in the dataset)."""
    rows = []
    with open(filepath) as fh:
        for line in fh:
            s = line.strip()
            if (not s or s.startswith('%')
                    or s.lower().startswith('depth') or s.startswith('vti')):
                continue
            try:
                vals = [float(x) for x in s.split()]
                if len(vals) == 6:
                    rows.append(vals)
            except ValueError:
                continue
    return pd.DataFrame(rows, columns=['depth', 'Vp', 'Vs', 'rho', 'GR', 'nphi'])


def load_well(data_dir):
    """Load main well log."""
    return _parse_txt(os.path.join(data_dir, 'well_2.txt'))


def load_facies(data_dir):
    """Load all per-facies text files; return dict: facies name → set of depths."""
    patterns = {
        'Clean Sand'   : 'well2_clnSand*.txt',
        'Cemented Sand': 'well2_cemSand*.txt',
        'Silty Sand 1' : 'well2_sltSand1*.txt',
        'Silty Sand 2' : 'well2_sltSand2*.txt',
        'Silty Shale'  : 'well2_sltShale*.txt',
        'Shale'        : 'well2_Shale*.txt',
    }
    result = {}
    for name, pat in patterns.items():
        files = glob.glob(os.path.join(data_dir, pat))
        if not files:
            print(f"  WARNING: no files matched {pat}")
            continue
        frames = [_parse_txt(f) for f in files]
        combined = pd.concat(frames, ignore_index=True)
        result[name] = set(np.round(combined['depth'].values, 1))
    return result


def assign_facies(well, facies_depths):
    """Label each depth sample by its facies (depth-matched)."""
    w = well.copy()
    w['facies'] = 'Background'
    d_r = w['depth'].round(1)
    for name, depths in facies_depths.items():
        w.loc[d_r.isin(depths), 'facies'] = name
    return w

### Execute: Load Well Data and Assign Facies

Parse the master log file then overlay the per-facies depth populations to
label every depth sample with its lithology class. Printing the facies counts
is a quick sanity-check that all six classes loaded correctly.

In [ ]:
DATA_DIR = os.path.join(os.getcwd(), 'data')

print("[1] Loading well data ...")
well = load_well(DATA_DIR)
print(f"    {len(well)} samples, depth {well['depth'].min():.0f}–{well['depth'].max():.0f} m")
print(f"    Columns: {list(well.columns)}")

print("\n[2] Assigning facies ...")
facies_depths = load_facies(DATA_DIR)
well = assign_facies(well, facies_depths)
print(well['facies'].value_counts().to_string(header=False))

well.head()

## 4. Batzle-Wang Live-Oil Fluid Properties

Accurate fluid substitution requires the **in-situ bulk modulus of the pore fluid**
at reservoir temperature and pressure. Batzle & Wang (1992) provide empirical
equations derived from laboratory measurements on a wide range of fluids.

**For live oil the procedure is:**
1. Dead-oil surface density from API gravity
2. Formation volume factor B₀ (accounts for dissolved gas swelling the oil)
3. Live-oil density at reservoir T and P
4. Dead-oil P-wave velocity at T (°C) and P (MPa) — B-W Eq. 20
5. Live-oil velocity via dissolved-gas correction (substitute live density into Eq. 20)
6. K_oil = ρ_live × (Vp_live / 1000)²  [GPa]

> **Unit note:** GOR for this well is in metric (sm³/sm³). B-W equations use
> imperial (scf/stb), so GOR is converted by × 5.615 inside the function.

In [ ]:
def batzle_wang_oil(T_c, P_mpa, API, GOR_sm3):
    """
    Batzle & Wang (1992) live-oil bulk modulus at reservoir conditions.
    Returns (K_oil GPa, rho_live g/cc, Vp_live m/s).
    """
    G_g = 0.6  # dissolved-gas gravity (relative to air)
    rho_0 = 141.5 / (131.5 + API)               # surface dead-oil density
    R_s   = GOR_sm3 * 5.615                      # GOR: sm³/sm³ → scf/stb
    T_f   = T_c * 9.0 / 5.0 + 32.0             # °C → °F
    B_0   = 0.972 + 0.00038 * (2.4 * R_s * np.sqrt(G_g / rho_0) + T_f + 17.8)**1.175
    rho_live = (rho_0 + 0.001224 * G_g * R_s) / B_0

    def _dead_vp(rho):                           # B-W Eq. 20 [m/s]
        return (2096.0 * np.sqrt(rho / (2.6 - rho))
                - 3.7 * T_c + 4.64 * P_mpa
                + 0.0115 * (4.12 * np.sqrt(1.08 / rho - 1.0) - 1.0) * T_c * P_mpa)

    Vp_live = _dead_vp(rho_live)                 # B-W Eq. 22: use live density
    K_oil   = rho_live * (Vp_live / 1000.0)**2  # [GPa]
    return K_oil, rho_live, Vp_live

### Execute: Compute Oil Properties

Apply the Batzle-Wang equations at the Glitne Field reservoir conditions.
The B-W live density can differ from the measured reservoir value for high-GOR
oils, so we use the **given density** (0.78 g/cc) to compute K_oil while still
relying on the B-W velocity equation.

In [ ]:
print("[0] Computing oil properties (Batzle-Wang 1992) ...")
K_OIL_bw, rho_oil_bw, Vp_oil_mps = batzle_wang_oil(
    RESERVOIR_TEMP, RESERVOIR_P, OIL_API, OIL_GOR)

# Use given density (0.78 g/cc) because B-W live density is unreliable
# for this high-GOR case; keep B-W velocity.
K_OIL = RHO_OIL * (Vp_oil_mps / 1000.0)**2

print(f"  B-W live density:     {rho_oil_bw:.3f} g/cc  (given: {RHO_OIL:.2f} g/cc)")
print(f"  B-W live velocity:    {Vp_oil_mps:.0f} m/s")
print(f"  K_oil (adopted):      {K_OIL:.3f} GPa  [= {RHO_OIL:.2f} × ({Vp_oil_mps/1000:.3f})²]")
print(f"  K_gas:                {K_GAS:.3f} GPa  |  K_brine: {K_BR:.3f} GPa")
print()
print("Fluid stiffness ranking (stiffer fluids → higher Vp after substitution):")
print(f"  Gas ({K_GAS}) << Oil ({K_OIL:.3f}) < Brine ({K_BR})  [GPa]")

## 5. Core Rock Physics Functions

Four fundamental functions underpin the entire substitution workflow:

### Voigt-Reuss-Hill (VRH) Mixing
`vrh(f, M1, M2)` — the arithmetic mean of the Voigt upper bound and Reuss lower
bound for a two-component mineral mixture. Used to compute the mineral frame
modulus K₀ (and G₀) as a function of clay volume Vsh.

### Gassmann Inversion
`gassmann_inv` — extracts the normalised dry-frame bulk modulus Kd/K₀ from the
saturated log-derived modulus. Closed-form inverse (Simm 2007 Eq. 4):
$$x = \frac{y\beta - 1}{y + \beta - 2}, \quad \beta = \frac{\phi K_0}{K_{fl}} + 1 - \phi, \quad y = \frac{K_{sat}}{K_0}$$

### Gassmann Forward
`gassmann_fwd` — predicts the new saturated modulus after replacing the pore fluid.
Shear modulus **μ is fluid-independent** — only Ksat changes.

### Poisson's Ratio
`poisson(Vp, Vs)` — the most sensitive seismic fluid indicator; used in crossplots.

In [ ]:
def vrh(f, M1, M2):
    """Voigt-Reuss-Hill average. f = volume fraction of M1."""
    Mv = f * M1 + (1 - f) * M2
    Mr = 1.0 / (f / M1 + (1 - f) / M2)
    return 0.5 * (Mv + Mr)


def gassmann_inv(Ksat, K0, Kfl, phi):
    """Invert Gassmann for normalised dry bulk modulus Kd/K0 (Simm 2007 Eq. 4)."""
    beta = phi * K0 / Kfl + 1.0 - phi
    y    = Ksat / K0
    return (y * beta - 1.0) / (y + beta - 2.0)


def gassmann_fwd(Kd_K0, K0, Kfl_new, phi):
    """Forward Gassmann: new saturated bulk modulus (GPa)."""
    x    = Kd_K0
    beta = phi * K0 / Kfl_new + 1.0 - phi
    return K0 * (x + (1.0 - x)**2 / (beta - x))


def poisson(Vp, Vs):
    """Poisson's ratio from Vp and Vs."""
    return (Vp**2 - 2*Vs**2) / (2*(Vp**2 - Vs**2))


print("Core rock physics functions defined.")

## 6. Rock Physics Pipeline — Steps 2–4

`compute_rock_physics` chains the first four computational steps of Simm's
workflow into a single function that enriches the well DataFrame:

| Output column | Derivation |
|---------------|-----------|
| `Vsh` | Linear GR normalisation between 5th / 95th percentile end-points |
| `phi` | Density-porosity: (ρ_mineral − ρ_log) / (ρ_mineral − ρ_brine) |
| `K0`, `G0` | VRH mineral frame moduli (quartz + shale mixture) |
| `mu`, `Ksat` | Saturated moduli from log velocities and density |
| `Kd_K0` | Gassmann inversion — assumes brine saturation throughout |
| `Kphi_K0` | Normalised pore stiffness = Kd/K₀ / (1 − Kd/K₀) |
| `PR`, `AI`, `VpVs` | Derived seismic quantities |

> **Unit note:** ρ [g/cc] × V² [km/s]² = GPa — no conversion needed.

In [ ]:
def compute_rock_physics(well):
    """Steps 2-4: Vsh, porosity, VRH moduli, Gassmann inversion, derived quantities."""
    w = well.copy()

    # ── Vsh from GR (data-driven end-points) ─────────────────────────────────
    gr_clean = np.percentile(w['GR'], 5)
    gr_shale = np.percentile(w['GR'], 95)
    w['Vsh'] = ((w['GR'] - gr_clean) / (gr_shale - gr_clean)).clip(0.0, 1.0)

    # ── Effective porosity from density ──────────────────────────────────────
    rho_min  = w['Vsh'] * RHO_SH + (1.0 - w['Vsh']) * RHO_QTZ
    w['phi'] = ((rho_min - w['rho']) / (rho_min - RHO_BR)).clip(0.01, 0.55)

    # ── Mineral moduli via VRH ────────────────────────────────────────────────
    w['K0'] = vrh(w['Vsh'], K_SH, K_QTZ)
    w['G0'] = vrh(w['Vsh'], G_SH, G_QTZ)

    # ── Saturated moduli from logs (ρ [g/cc] × V² [km/s]² = GPa) ─────────────
    w['mu']   = w['rho'] * w['Vs']**2
    w['Ksat'] = w['rho'] * w['Vp']**2 - (4.0/3.0) * w['mu']

    # ── Gassmann inversion (assumes brine saturation) ─────────────────────────
    w['Kd_K0'] = gassmann_inv(w['Ksat'], w['K0'], K_BR, w['phi'])
    w['Kd']    = w['Kd_K0'] * w['K0']

    # Normalised pore stiffness  Kφ/K0 = (Kd/K0) / (1 − Kd/K0)
    denom = (1.0 - w['Kd_K0']).where(lambda x: x.abs() > 1e-4, other=np.nan)
    w['Kphi_K0'] = w['Kd_K0'] / denom

    # ── Derived seismic quantities ────────────────────────────────────────────
    w['PR']   = poisson(w['Vp'], w['Vs'])
    w['AI']   = w['rho'] * w['Vp']
    w['VpVs'] = w['Vp'] / w['Vs']

    return w

### Execute: Compute Rock Physics

Run the pipeline on the loaded well. The Kd/K₀ statistics give an early
indication of data quality — negative values in shaly or low-porosity zones
are expected when the brine-saturation assumption is locally violated.

In [ ]:
print("[3] Computing rock physics ...")
well = compute_rock_physics(well)

print(f"  Porosity range:   {well['phi'].min():.3f} – {well['phi'].max():.3f} v/v")
print(f"  Kd/K0  P5–P95:   {well['Kd_K0'].quantile(0.05):.3f} – {well['Kd_K0'].quantile(0.95):.3f}")
print(f"  (negative Kd/K0 expected in shaly / low-φ zones)")
print()
# Facies summary of key properties
print(well.groupby('facies')[['phi', 'Vp', 'Vs', 'Kd_K0']].mean().round(3).to_string())

## 7. Plotting Utilities

### Facies colour scheme
Each lithology class is assigned a distinct colour used consistently across every
figure. Warm yellows/oranges = sands; greens = silty sands; greys = shaly facies.

### `_shade_facies`
Adds translucent colour bands behind any depth-track axis to indicate facies
intervals — a standard wireline log convention.

### `_add_reservoir_markers`
Draws dashed horizontal lines at Top Heimdal (2153 m) and the OWC (2183 m) on
every axis in a figure. These are the key boundaries for the Glitne reservoir.

In [ ]:
FACIES_COLORS = {
    'Clean Sand'   : '#FFD700',
    'Cemented Sand': '#FF8C00',
    'Silty Sand 1' : '#6DBF67',
    'Silty Sand 2' : '#2E8B57',
    'Silty Shale'  : '#AAAAAA',
    'Shale'        : '#606060',
    'Background'   : '#E0E0E0',
}


def _shade_facies(ax, w):
    """Add subtle facies colour bands behind a depth track."""
    depth = w['depth'].values
    fac   = w['facies'].values
    i = 0
    while i < len(depth):
        j = i
        while j < len(depth) and fac[j] == fac[i]:
            j += 1
        ax.axhspan(depth[i], depth[min(j, len(depth)-1)],
                   color=FACIES_COLORS.get(fac[i], '#FFFFFF'), alpha=0.18, linewidth=0)
        i = j


def _add_reservoir_markers(axes, depth_col):
    """Add Top Heimdal and OWC horizontal marker lines to all axes."""
    for ax in axes:
        ax.axhline(TOP_HEIMDAL, color='#2CA02C', ls='-.', lw=0.9, alpha=0.8)
        ax.axhline(OWC_DEPTH,   color='#1F77B4', ls='-.', lw=0.9, alpha=0.8)


print("Plotting utilities defined.")

## 8. Dry Rock Trend Fitting — Simm's Adaptive Conditioning (Step 5)

This is the **central methodological contribution** of Simm (2007).

**Problem:** Gassmann inversion of shaley or poorly consolidated sands often
produces Kd/K₀ values that are negative or otherwise unphysical — not because
the physics is wrong, but because Vs logs in mixed lithologies are noisy and
the 100% brine-saturation assumption may be locally invalid.

**Solution:** Fit a 2nd-order polynomial to the **clean-sand** population in
Kd/K₀ vs φ space, anchored at the mineral point (φ = 0, Kd/K₀ = 1):
$$K_d/K_0 = a\phi^2 + b\phi + 1$$
This trend is then applied **only to silty sands** (the problematic intermediate
lithologies). Clean sands, cemented sands, and shales keep their raw inverted
Kd/K₀ values.

In [ ]:
def fit_dry_rock_trend(well):
    """
    Fit Kd/K0 = aφ² + bφ + 1 to sand data, anchored at mineral point (φ=0).
    Returns coefficients [a, b, 1.0] for np.polyval.
    """
    sand_facies = ['Clean Sand', 'Cemented Sand', 'Silty Sand 1', 'Silty Sand 2']
    mask = (
        well['facies'].isin(sand_facies)
        & (well['Kd_K0'] > 0.05) & (well['Kd_K0'] < 1.00)
        & (well['phi']   > 0.05)
    )
    sand = well[mask]
    if len(sand) < 5:
        print("  WARNING: very few valid sand points for trend fitting.")
    phi = sand['phi'].values
    y   = sand['Kd_K0'].values - 1.0   # shift so anchor = 0
    X, _, _, _ = np.linalg.lstsq(np.column_stack([phi**2, phi]), y, rcond=None)
    a, b = X
    return np.array([a, b, 1.0])

### Execute: Fit the Dry-Rock Trend

Run the least-squares fit and print the coefficients. The polynomial should
pass through (φ=0, Kd/K₀=1) by construction and curve downward with increasing
porosity — stiffer at lower porosity (cemented) and more compliant at higher
porosity (clean loose sand).

In [ ]:
print("[4] Fitting conditioned dry-rock trend ...")
coeffs = fit_dry_rock_trend(well)
print(f"  Kd/K0 = {coeffs[0]:.3f}·φ² + {coeffs[1]:.3f}·φ + {coeffs[2]:.3f}")
print("  Conditioning strategy:")
print("    • Clean Sand & Cemented Sand: raw Kd/K0 (these are the reference facies)")
print("    • Silty Sand 1 & 2:           conditioned polynomial trend")
print("    • Shales:                     raw Kd/K0 (too different from sand trend)")

### Figure 2 — Kd/K₀ vs Porosity Template

This is the **central QC diagnostic** from Simm (2007), Figs 5 & 8.

- **X-axis:** effective porosity φ_e
- **Y-axis:** normalised dry bulk modulus Kd/K₀
  (0 = fully compliant framework; 1 = rigid mineral)
- **Horizontal dashed lines:** contours of constant pore stiffness
  Kφ/K₀ = 0.1, 0.2, … 0.5 (horizontal because Kφ/K₀ depends only on Kd/K₀)
- **Black star:** mineral point anchor (φ = 0, Kd/K₀ = 1)
- **Black curve:** fitted polynomial dry-rock trend

Inspect this plot before proceeding to fluid substitution. Clean sands should
cluster at high Kd/K₀; shales and silty intervals often scatter low or negative.
The trend line should bisect the clean-sand cloud and pass through the mineral point.

In [ ]:
def fig_kd_k0(w, coeffs):
    """Kd/K0 vs porosity template — central diagnostic from Simm (2007)."""
    fig, ax = plt.subplots(figsize=(9, 7))
    phi_r = np.linspace(0.01, 0.50, 300)

    for c in [0.1, 0.2, 0.3, 0.4, 0.5]:
        kd_k0_c = c / (1.0 + c)
        ax.axhline(kd_k0_c, color='#888888', ls='--', lw=0.9, alpha=0.7)
        ax.text(0.505, kd_k0_c + 0.007, f'Kφ/K₀ = {c:.1f}',
                fontsize=8, color='#555555', va='bottom')

    order = ['Background', 'Silty Shale', 'Shale',
             'Silty Sand 2', 'Silty Sand 1', 'Cemented Sand', 'Clean Sand']
    for fac in order:
        grp   = w[w['facies'] == fac]
        valid = grp[(grp['Kd_K0'] > -0.35) & (grp['Kd_K0'] < 1.05)]
        ax.scatter(valid['phi'], valid['Kd_K0'],
                   c=FACIES_COLORS.get(fac, '#DDDDDD'),
                   s=10, alpha=0.55, edgecolors='none', label=fac, zorder=3)

    ax.plot(phi_r, np.polyval(coeffs, phi_r), 'k-', lw=2.5, zorder=6,
            label='Conditioned dry-rock trend')
    ax.scatter([0.0], [1.0], marker='*', s=200, c='black',
               zorder=7, label='Mineral point (φ=0)')
    ax.axhline(0.0, color='black', lw=0.8, ls=':')
    ax.set_xlim(-0.01, 0.52); ax.set_ylim(-0.35, 1.08)
    ax.set_xlabel('Effective Porosity  φ_e  (v/v)', fontsize=12)
    ax.set_ylabel('Normalised Dry Bulk Modulus  K_d / K₀', fontsize=12)
    ax.set_title(
        'Dry Rock Bulk Modulus Template  (Simm 2007, Figs 5 & 8)\n'
        'Dashed lines = constant Kφ/K₀ contours  |  '
        f'K_qtz={K_QTZ} GPa, K_clay={K_SH} GPa, G_clay={G_SH} GPa', fontsize=10)
    ax.legend(fontsize=8, markerscale=1.8, loc='upper right')
    ax.grid(True, alpha=0.25)
    return fig

fig2 = fig_kd_k0(well, coeffs)
plt.show()

## 9. Fluid Substitution Functions — Step 6

Two parallel forward-substitution functions enable a direct method comparison:

### `apply_fluid_substitution` — Simm's Adaptive Approach
Applies the conditioned dry-rock trend **only to silty sands**. Clean/cemented
sands and shales use their raw inverted Kd/K₀ (the reference facies).

### `apply_default_fluid_substitution` — Baseline
Uses the **raw inverted Kd/K₀** (clipped to [0.05, 0.99]) for every facies.
This is standard Gassmann without adaptive conditioning.

**Common outputs** (suffix = `'oil'` or `'gas'`):

| Column | Quantity |
|--------|----------|
| `Ksat_{suffix}` | New saturated bulk modulus (GPa) |
| `rho_{suffix}` | Updated density (brine mass replaced) |
| `Vp_{suffix}`, `Vs_{suffix}` | New velocities — note μ is fluid-independent |
| `PR_{suffix}`, `AI_{suffix}` | Poisson's ratio and acoustic impedance |

In [ ]:
def apply_fluid_substitution(well, coeffs, K_fl_new, RHO_fl_new, suffix):
    """Simm adaptive Gassmann: applies dry-rock trend selectively to silty sands."""
    w = well.copy()
    conditioned = ['Silty Sand 1', 'Silty Sand 2']
    w['Kd_K0_c'] = w['Kd_K0'].where(
        ~w['facies'].isin(conditioned),
        np.polyval(coeffs, w['phi']).clip(0.05, 0.99))

    w[f'Ksat_{suffix}'] = gassmann_fwd(w['Kd_K0_c'], w['K0'], K_fl_new, w['phi'])
    w[f'rho_{suffix}']  = w['rho'] - w['phi'] * RHO_BR + w['phi'] * RHO_fl_new
    w[f'Vp_{suffix}']   = np.sqrt(
        (w[f'Ksat_{suffix}'] + (4.0/3.0)*w['mu']) / w[f'rho_{suffix}'])
    w[f'Vs_{suffix}']   = np.sqrt(w['mu'] / w[f'rho_{suffix}'])
    w[f'PR_{suffix}']   = poisson(w[f'Vp_{suffix}'], w[f'Vs_{suffix}'])
    w[f'AI_{suffix}']   = w[f'rho_{suffix}'] * w[f'Vp_{suffix}']
    return w


def apply_default_fluid_substitution(well, K_fl_new, RHO_fl_new, suffix):
    """Baseline Gassmann: raw Kd/K0 (no conditioning) for all facies."""
    w = well.copy()
    w['Kd_K0_raw'] = w['Kd_K0'].clip(0.05, 0.99)

    w[f'Ksat_default_{suffix}'] = gassmann_fwd(w['Kd_K0_raw'], w['K0'], K_fl_new, w['phi'])
    w[f'rho_default_{suffix}']  = w['rho'] - w['phi'] * RHO_BR + w['phi'] * RHO_fl_new
    w[f'Vp_default_{suffix}']   = np.sqrt(
        (w[f'Ksat_default_{suffix}'] + (4.0/3.0)*w['mu']) / w[f'rho_default_{suffix}'])
    w[f'Vs_default_{suffix}']   = np.sqrt(w['mu'] / w[f'rho_default_{suffix}'])
    w[f'PR_default_{suffix}']   = poisson(w[f'Vp_default_{suffix}'], w[f'Vs_default_{suffix}'])
    w[f'AI_default_{suffix}']   = w[f'rho_default_{suffix}'] * w[f'Vp_default_{suffix}']
    return w

### Execute: Apply Fluid Substitutions

Run both the Simm conditioned method and the default method for oil and gas.
Then print mean velocity and Poisson's ratio shifts for the sand facies, plus
per-facies differences between the two methods — non-zero differences appear
only in the silty-sand facies where conditioning is applied.

In [ ]:
print("[5] Applying fluid substitutions ...")

# Simm conditioned substitution
well = apply_fluid_substitution(well, coeffs, K_OIL,  RHO_OIL,  suffix='oil')
well = apply_fluid_substitution(well, coeffs, K_GAS,  RHO_GAS,  suffix='gas')

# Default substitution (raw Kd/K0 — for comparison)
well = apply_default_fluid_substitution(well, K_OIL, RHO_OIL, suffix='oil')
well = apply_default_fluid_substitution(well, K_GAS, RHO_GAS, suffix='gas')

sand = well[well['facies'].isin(['Clean Sand','Cemented Sand','Silty Sand 1','Silty Sand 2'])]
print(f"  Mean ΔVp  brine→oil (sand): {(sand['Vp_oil']  - sand['Vp']).mean():.3f} km/s")
print(f"  Mean ΔVp  brine→gas (sand): {(sand['Vp_gas']  - sand['Vp']).mean():.3f} km/s")
print(f"  Mean ΔPR  brine→oil (sand): {(sand['PR_oil']  - sand['PR']).mean():.3f}")
print(f"  Mean ΔPR  brine→gas (sand): {(sand['PR_gas']  - sand['PR']).mean():.3f}")
print()
print("  Default vs Simm differences (by facies):")
for fac in ['Clean Sand','Cemented Sand','Silty Sand 1','Silty Sand 2','Silty Shale']:
    grp = well[well['facies'] == fac]
    if len(grp) == 0: continue
    dvp_oil = (grp['Vp_default_oil'] - grp['Vp_oil']).mean()
    dvp_gas = (grp['Vp_default_gas'] - grp['Vp_gas']).mean()
    dpr_oil = (grp['PR_default_oil'] - grp['PR_oil']).mean()
    print(f"    {fac}: ΔVp_oil={dvp_oil:.3f}, ΔVp_gas={dvp_gas:.3f}, ΔPR_oil={dpr_oil:.3f}")

## 10. Figure 1 — Full Well Log Display (8-Track)

With the fluid substitution complete, we can now display the full log suite
showing all three fluid scenarios side by side. The **eight tracks** are:

1. **Depth ruler** — depth labels in metres
2. **GR** — gamma-ray, the primary shale indicator (GAPI)
3. **Vsh** — shale volume derived from GR normalisation
4. **φ_e** — effective porosity from density log
5. **ρ** — bulk density
6. **Vp** — P-wave velocity for all three fluids + both methods
7. **Vs** — S-wave velocity (same colour scheme)
8. **Poisson's ratio** — the most sensitive fluid indicator (reference line at σ = 0.33)

Colour-shaded facies bands and Top Heimdal / OWC markers give geological context.

In [ ]:
def fig_well_logs(w):
    fig = plt.figure(figsize=(22, 13))
    gs  = GridSpec(1, 8, figure=fig, wspace=0.06,
                   left=0.04, right=0.97, top=0.88, bottom=0.10)
    depth = w['depth']

    def new_ax(col, sharey=None):
        return fig.add_subplot(gs[0, col], sharey=sharey)

    ax_d = new_ax(0)
    ax0  = new_ax(1, ax_d); ax1 = new_ax(2, ax_d); ax2 = new_ax(3, ax_d)
    ax3  = new_ax(4, ax_d); ax4 = new_ax(5, ax_d); ax5 = new_ax(6, ax_d); ax6 = new_ax(7, ax_d)
    axes = [ax_d, ax0, ax1, ax2, ax3, ax4, ax5, ax6]

    for ax in axes:
        ax.invert_yaxis(); ax.set_ylim(depth.max()+5, depth.min()-5)
        ax.tick_params(labelsize=7); ax.grid(axis='x', color='#CCCCCC', lw=0.4)
        _shade_facies(ax, w)
    for ax in axes[1:]:
        ax.set_yticks([])

    _add_reservoir_markers(axes, depth)
    xlim = ax_d.get_xlim()
    ax_d.text(xlim[1]*0.5, TOP_HEIMDAL-3, 'Top Heimdal',
              va='bottom', ha='center', fontsize=6, color='#2CA02C', style='italic')
    ax_d.text(xlim[1]*0.5, OWC_DEPTH-3,   'OWC',
              va='bottom', ha='center', fontsize=6, color='#1F77B4', style='italic')

    ax_d.set_xlim(-0.5, 0.5); ax_d.set_xticks([])
    ax_d.set_ylabel('Depth (m)', fontsize=9, fontweight='bold')
    for d in np.arange(depth.min(), depth.max()+50, 50):
        ax_d.text(0, d, f'{d:.0f}', ha='center', va='center', fontsize=6)

    ax0.plot(w['GR'],  depth, 'k-', lw=0.6); ax0.set_xlim(0, 150)
    ax0.set_xlabel('GR\n(GAPI)', fontsize=8)

    ax1.fill_betweenx(depth, 0, w['Vsh'], color='sienna', alpha=0.5)
    ax1.plot(w['Vsh'], depth, color='sienna', lw=0.6); ax1.set_xlim(0, 1)
    ax1.set_xlabel('Vsh\n(v/v)', fontsize=8)

    ax2.fill_betweenx(depth, 0, w['phi'], color='teal', alpha=0.4)
    ax2.plot(w['phi'], depth, color='teal', lw=0.6); ax2.set_xlim(0, 0.50)
    ax2.set_xlabel('φ_e\n(v/v)', fontsize=8)

    ax3.plot(w['rho'], depth, color='#8B0000', lw=0.8); ax3.set_xlim(1.8, 2.8)
    ax3.set_xlabel('ρ\n(g/cc)', fontsize=8)

    for ax, col, lbl, xlim in [
        (ax4, 'Vp', 'Vp\n(km/s)', (1.2, 4.5)),
        (ax5, 'Vs', 'Vs\n(km/s)', (0.4, 2.5)),
        (ax6, 'PR', "Poisson's\nratio", (0.0, 0.50)),
    ]:
        ax.plot(w[col],                depth, 'b-',  lw=1.0, label='Brine')
        ax.plot(w[f'{col}_oil'],       depth, color='#2CA02C', ls='--', lw=1.0, label='Oil (Simm)', alpha=0.85)
        ax.plot(w[f'{col}_gas'],       depth, 'r:',  lw=1.2, label='Gas (Simm)', alpha=0.85)
        ax.plot(w[f'{col}_default_oil'], depth, color='#FF4500', ls='-.', lw=1.0, label='Oil (Default)', alpha=0.7)
        ax.plot(w[f'{col}_default_gas'], depth, color='#8B0000', ls='-.', lw=1.2, label='Gas (Default)', alpha=0.7)
        ax.set_xlim(*xlim); ax.set_xlabel(lbl, fontsize=8)
        ax.legend(fontsize=5, loc='lower right', ncol=2)
    ax6.axvline(0.33, color='gray', ls=':', lw=0.8)

    patches = [mpatches.Patch(color=c, alpha=0.7, label=f)
               for f, c in FACIES_COLORS.items() if f != 'Background']
    patches += [mpatches.Patch(color='#2CA02C', alpha=0.8, label='Top Heimdal'),
                mpatches.Patch(color='#1F77B4', alpha=0.8, label='OWC')]
    fig.legend(handles=patches, loc='lower center', ncol=len(patches),
               fontsize=7, title='Facies / Horizons', bbox_to_anchor=(0.52, 0.01))
    fig.suptitle(
        'Well 2 — Glitne Field, Norway  (Heimdal Formation)\n'
        'Brine (blue) · Oil Simm (green --) · Gas Simm (red ·) · '
        'Oil Default (orange -·) · Gas Default (brown -·)',
        fontsize=11, fontweight='bold')
    return fig

fig1 = fig_well_logs(well)
plt.show()

## 11. Figure 1b — Zoomed Well Log (2120–2320 m)

Same 8-track layout restricted to the **reservoir interval**, with 10 m depth
labels instead of 50 m. This zoom resolves the fluid-substitution response
across the Heimdal Formation oil column and the water leg below the OWC (2183 m).

In [ ]:
def fig_well_logs_zoomed(w, depth_min=2120.0, depth_max=2320.0):
    """Zoomed 8-track display over the reservoir interval."""
    fig = plt.figure(figsize=(22, 13))
    gs  = GridSpec(1, 8, figure=fig, wspace=0.06,
                   left=0.04, right=0.97, top=0.88, bottom=0.10)
    depth = w['depth']
    zm    = (depth >= depth_min) & (depth <= depth_max)
    wz    = w[zm].copy()
    dz    = depth[zm]

    def new_ax(col, sharey=None):
        return fig.add_subplot(gs[0, col], sharey=sharey)

    ax_d = new_ax(0)
    ax0  = new_ax(1, ax_d); ax1 = new_ax(2, ax_d); ax2 = new_ax(3, ax_d)
    ax3  = new_ax(4, ax_d); ax4 = new_ax(5, ax_d); ax5 = new_ax(6, ax_d); ax6 = new_ax(7, ax_d)
    axes = [ax_d, ax0, ax1, ax2, ax3, ax4, ax5, ax6]

    for ax in axes:
        ax.invert_yaxis(); ax.set_ylim(depth_max+2, depth_min-2)
        ax.tick_params(labelsize=7); ax.grid(axis='x', color='#CCCCCC', lw=0.4)
        _shade_facies(ax, wz)
    for ax in axes[1:]:
        ax.set_yticks([])

    _add_reservoir_markers(axes, depth)
    xlim = ax_d.get_xlim()
    if depth_min <= TOP_HEIMDAL <= depth_max:
        ax_d.text(xlim[1]*0.5, TOP_HEIMDAL-1.5, 'Top Heimdal',
                  va='bottom', ha='center', fontsize=6, color='#2CA02C', style='italic')
    if depth_min <= OWC_DEPTH <= depth_max:
        ax_d.text(xlim[1]*0.5, OWC_DEPTH-1.5, 'OWC',
                  va='bottom', ha='center', fontsize=6, color='#1F77B4', style='italic')

    ax_d.set_xlim(-0.5, 0.5); ax_d.set_xticks([])
    ax_d.set_ylabel('Depth (m)', fontsize=9, fontweight='bold')
    for d in np.arange(depth_min, depth_max+10, 10):
        ax_d.text(0, d, f'{d:.0f}', ha='center', va='center', fontsize=6)

    ax0.plot(wz['GR'], dz, 'k-', lw=0.6); ax0.set_xlim(0, 150)
    ax0.set_xlabel('GR\n(GAPI)', fontsize=8)

    ax1.fill_betweenx(dz, 0, wz['Vsh'], color='sienna', alpha=0.5)
    ax1.plot(wz['Vsh'], dz, color='sienna', lw=0.6); ax1.set_xlim(0, 1)
    ax1.set_xlabel('Vsh\n(v/v)', fontsize=8)

    ax2.fill_betweenx(dz, 0, wz['phi'], color='teal', alpha=0.4)
    ax2.plot(wz['phi'], dz, color='teal', lw=0.6); ax2.set_xlim(0, 0.50)
    ax2.set_xlabel('φ_e\n(v/v)', fontsize=8)

    ax3.plot(wz['rho'], dz, color='#8B0000', lw=0.8); ax3.set_xlim(1.8, 2.8)
    ax3.set_xlabel('ρ\n(g/cc)', fontsize=8)

    for ax, col, lbl, xlim in [
        (ax4, 'Vp', 'Vp\n(km/s)', (1.2, 4.5)),
        (ax5, 'Vs', 'Vs\n(km/s)', (0.4, 2.5)),
        (ax6, 'PR', "Poisson's\nratio", (0.0, 0.50)),
    ]:
        ax.plot(wz[col],                dz, 'b-',  lw=1.0, label='Brine')
        ax.plot(wz[f'{col}_oil'],       dz, color='#2CA02C', ls='--', lw=1.0, label='Oil (Simm)', alpha=0.85)
        ax.plot(wz[f'{col}_gas'],       dz, 'r:',  lw=1.2, label='Gas (Simm)', alpha=0.85)
        ax.plot(wz[f'{col}_default_oil'], dz, color='#FF4500', ls='-.', lw=1.0, label='Oil (Default)', alpha=0.7)
        ax.plot(wz[f'{col}_default_gas'], dz, color='#8B0000', ls='-.', lw=1.2, label='Gas (Default)', alpha=0.7)
        ax.set_xlim(*xlim); ax.set_xlabel(lbl, fontsize=8)
        ax.legend(fontsize=5, loc='lower right', ncol=2)
    ax6.axvline(0.33, color='gray', ls=':', lw=0.8)

    patches = [mpatches.Patch(color=c, alpha=0.7, label=f)
               for f, c in FACIES_COLORS.items() if f != 'Background']
    patches += [mpatches.Patch(color='#2CA02C', alpha=0.8, label='Top Heimdal'),
                mpatches.Patch(color='#1F77B4', alpha=0.8, label='OWC')]
    fig.legend(handles=patches, loc='lower center', ncol=len(patches),
               fontsize=7, title='Facies / Horizons', bbox_to_anchor=(0.52, 0.01))
    fig.suptitle(
        f'Well 2 — Glitne Field  [ZOOMED: {depth_min:.0f}–{depth_max:.0f} m]\n'
        'Brine (blue) · Oil Simm (green --) · Gas Simm (red ·) · '
        'Oil Default (orange -·) · Gas Default (brown -·)',
        fontsize=11, fontweight='bold')
    return fig

fig1b = fig_well_logs_zoomed(well)
plt.show()

## 12. Figure 3 — Rock Physics Crossplot Suite

Four complementary crossplots, each coloured by facies and showing brine (circles),
oil-substituted (triangles), and gas-substituted (squares) populations:

| Panel | Axes | Purpose |
|-------|------|---------|
| (a) | Vp vs φ_e | Velocity-porosity; cement vs sorting effect |
| (b) | Vp/Vs vs Vp | Combines lithology and fluid sensitivity |
| (c) | AI vs Vp/Vs | Standard AVO proxy — gas sands move lower-left |
| (d) | Kd/K₀ vs φ_e coloured by Vsh | Dry-rock frame; fluid-independent |

Together these four panels characterise the rock physics response before and
after substitution, and highlight which facies are most sensitive to fluid changes.

In [ ]:
def fig_crossplots(w):
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    ax1, ax2, ax3, ax4 = axes.flatten()
    labelled = w[w['facies'] != 'Background']
    kw_br  = dict(s=10, alpha=0.65, edgecolors='none', marker='o')
    kw_oil = dict(s=8,  alpha=0.45, edgecolors='none', marker='^')
    kw_gas = dict(s=8,  alpha=0.45, edgecolors='none', marker='s')

    # (a) Vp vs φ_e
    for fac, grp in labelled.groupby('facies'):
        c = FACIES_COLORS[fac]
        ax1.scatter(grp['phi'], grp['Vp'],     c=c, label=fac,          **kw_br)
        ax1.scatter(grp['phi'], grp['Vp_oil'], c=c, label='_nolegend_', **kw_oil)
        ax1.scatter(grp['phi'], grp['Vp_gas'], c=c, label='_nolegend_', **kw_gas)
    ax1.scatter([], [], marker='^', c='k', s=12, alpha=0.6, label='Oil-sub')
    ax1.scatter([], [], marker='s', c='k', s=12, alpha=0.6, label='Gas-sub')
    ax1.set_xlabel('Effective Porosity φ_e', fontsize=10)
    ax1.set_ylabel('Vp  (km/s)', fontsize=10)
    ax1.set_title('(a)  Vp vs Porosity', fontsize=11)
    ax1.legend(fontsize=6, markerscale=1.8, ncol=2); ax1.grid(True, alpha=0.25)

    # (b) Vp/Vs vs Vp
    for fac, grp in labelled.groupby('facies'):
        c = FACIES_COLORS[fac]
        ax2.scatter(grp['Vp'],     grp['VpVs'],                 c=c, label=fac,          **kw_br)
        ax2.scatter(grp['Vp_oil'], grp['Vp_oil']/grp['Vs_oil'], c=c, label='_nolegend_', **kw_oil)
        ax2.scatter(grp['Vp_gas'], grp['Vp_gas']/grp['Vs_gas'], c=c, label='_nolegend_', **kw_gas)
    ax2.scatter([], [], marker='^', c='k', s=12, alpha=0.6, label='Oil-sub')
    ax2.scatter([], [], marker='s', c='k', s=12, alpha=0.6, label='Gas-sub')
    ax2.set_xlabel('Vp  (km/s)', fontsize=10); ax2.set_ylabel('Vp/Vs  ratio', fontsize=10)
    ax2.set_title('(b)  Vp/Vs vs Vp', fontsize=11)
    ax2.axhline(2.0, color='gray', ls='--', lw=0.8, alpha=0.6, label='Vp/Vs = 2')
    ax2.legend(fontsize=6, markerscale=1.8, ncol=2); ax2.grid(True, alpha=0.25)

    # (c) AI vs Vp/Vs
    for fac, grp in labelled.groupby('facies'):
        c = FACIES_COLORS[fac]
        ax3.scatter(grp['AI'],     grp['VpVs'],                 c=c, label=fac,          **kw_br)
        ax3.scatter(grp['AI_oil'], grp['Vp_oil']/grp['Vs_oil'], c=c, label='_nolegend_', **kw_oil)
        ax3.scatter(grp['AI_gas'], grp['Vp_gas']/grp['Vs_gas'], c=c, label='_nolegend_', **kw_gas)
    ax3.scatter([], [], marker='^', c='k', s=12, alpha=0.6, label='Oil-sub')
    ax3.scatter([], [], marker='s', c='k', s=12, alpha=0.6, label='Gas-sub')
    ax3.set_xlabel('Acoustic Impedance  (g/cc · km/s)', fontsize=10)
    ax3.set_ylabel('Vp/Vs  ratio', fontsize=10)
    ax3.set_title('(c)  AI vs Vp/Vs  (AVO proxy)\nCircles=brine  △=oil-sub  □=gas-sub', fontsize=10)
    ax3.legend(fontsize=6, markerscale=1.8, ncol=2); ax3.grid(True, alpha=0.25)

    # (d) Kd/K0 vs φ coloured by Vsh
    valid = labelled[(labelled['Kd_K0'] > -0.35) & (labelled['Kd_K0'] < 1.05)]
    sc = ax4.scatter(valid['phi'], valid['Kd_K0'],
                     c=valid['Vsh'], cmap='RdYlGn_r', vmin=0, vmax=1,
                     s=10, alpha=0.65, edgecolors='none')
    plt.colorbar(sc, ax=ax4, label='Vsh  (v/v)', shrink=0.85)
    for c in [0.1, 0.2, 0.3, 0.4, 0.5]:
        ax4.axhline(c/(1+c), color='gray', ls='--', lw=0.7, alpha=0.6)
    ax4.axhline(0.0, color='black', lw=0.8, ls=':')
    ax4.set_xlim(-0.01, 0.52); ax4.set_ylim(-0.35, 1.05)
    ax4.set_xlabel('Effective Porosity φ_e', fontsize=10)
    ax4.set_ylabel('K_d / K₀', fontsize=10)
    ax4.set_title('(d)  K_d/K₀ vs Porosity  (coloured by Vsh)\n[Dry-rock — fluid independent]', fontsize=10)
    ax4.grid(True, alpha=0.25)

    fig.suptitle('Rock Physics Crossplot Suite — Well 2 (Glitne Field)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    return fig

fig3 = fig_crossplots(well)
plt.show()

## 13. Figure 4 — Fluid Substitution Sensitivity Arrows

Focussing on **sand facies only**, this figure draws arrows from each brine
sample to its oil- and gas-substituted counterpart:

- **Panel (a): AI vs Vp/Vs** — the AVO template. Gas substitution drives sands
  toward lower AI and higher Vp/Vs (class II/III gas sand territory). Oil shows
  a smaller but directionally similar shift.
- **Panel (b): AI vs Poisson's ratio** — the seismic inversion plane. Gas sands
  move sharply toward lower σ.

The spread of arrows by facies colour reveals which lithologies are most
sensitive to fluid changes — clean porous sands show the largest vectors.

In [ ]:
def fig_fluid_sensitivity(w):
    """Fluid substitution vectors: brine→oil (green) and brine→gas (red)."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    ax1, ax2 = axes
    sand_facies = ['Clean Sand', 'Cemented Sand', 'Silty Sand 1', 'Silty Sand 2']
    labelled    = w[w['facies'].isin(sand_facies)]
    kw_br  = dict(s=22, alpha=0.70, edgecolors='none', marker='o')
    kw_oil = dict(s=18, alpha=0.65, edgecolors='none', marker='^')
    kw_gas = dict(s=18, alpha=0.65, edgecolors='none', marker='s')

    for fac, grp in labelled.groupby('facies'):
        c = FACIES_COLORS[fac]
        ax1.scatter(grp['AI'],     grp['VpVs'],                 c=c, **kw_br)
        ax1.scatter(grp['AI_oil'], grp['Vp_oil']/grp['Vs_oil'], c=c, **kw_oil)
        ax1.scatter(grp['AI_gas'], grp['Vp_gas']/grp['Vs_gas'], c=c, **kw_gas)
        ax2.scatter(grp['PR'],     grp['AI'],     c=c, **kw_br)
        ax2.scatter(grp['PR_oil'], grp['AI_oil'], c=c, **kw_oil)
        ax2.scatter(grp['PR_gas'], grp['AI_gas'], c=c, **kw_gas)

    subset = labelled.sample(min(100, len(labelled)), random_state=42)
    for _, r in subset.iterrows():
        ax1.annotate('', xy=(r['AI_oil'], r['Vp_oil']/r['Vs_oil']),
                     xytext=(r['AI'], r['VpVs']),
                     arrowprops=dict(arrowstyle='->', color='#2CA02C', lw=0.7, alpha=0.45))
        ax2.annotate('', xy=(r['PR_oil'], r['AI_oil']), xytext=(r['PR'], r['AI']),
                     arrowprops=dict(arrowstyle='->', color='#2CA02C', lw=0.7, alpha=0.45))
        ax1.annotate('', xy=(r['AI_gas'], r['Vp_gas']/r['Vs_gas']),
                     xytext=(r['AI'], r['VpVs']),
                     arrowprops=dict(arrowstyle='->', color='#D62728', lw=0.7, alpha=0.45))
        ax2.annotate('', xy=(r['PR_gas'], r['AI_gas']), xytext=(r['PR'], r['AI']),
                     arrowprops=dict(arrowstyle='->', color='#D62728', lw=0.7, alpha=0.45))

    brine_pt = mpatches.Patch(color='gray',    label='Brine (circle)')
    oil_pt   = mpatches.Patch(color='#2CA02C', label='Oil-sub (triangle, green arrow)')
    gas_pt   = mpatches.Patch(color='#D62728', label='Gas-sub (square, red arrow)')
    ax1.set_xlabel('Acoustic Impedance  (g/cc · km/s)', fontsize=10)
    ax1.set_ylabel('Vp/Vs  ratio', fontsize=10)
    ax1.set_title('(a)  AI vs Vp/Vs\nBrine → Oil (green)  Brine → Gas (red)', fontsize=11)
    ax1.legend(handles=[brine_pt, oil_pt, gas_pt], fontsize=7); ax1.grid(True, alpha=0.25)
    ax2.set_xlabel("Poisson's ratio  σ", fontsize=10)
    ax2.set_ylabel('Acoustic Impedance  (g/cc · km/s)', fontsize=10)
    ax2.set_title("(b)  AI vs Poisson's ratio\nBrine → Oil (green)  Brine → Gas (red)", fontsize=11)
    ax2.legend(handles=[brine_pt, oil_pt, gas_pt], fontsize=7); ax2.grid(True, alpha=0.25)
    fig.suptitle('Fluid Substitution Effect — Sand Facies, Glitne Field WELL-2',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    return fig

fig4 = fig_fluid_sensitivity(well)
plt.show()

## 14. Figure 5 — Per-Facies Vp: Default vs Simm Conditioning

One depth-profile panel per sand/silty class. Each panel overlays five Vp curves:

| Style | Meaning |
|-------|---------|
| Blue solid | Observed brine Vp |
| Red dashed | Default oil (raw Kd/K₀) |
| Green solid | Simm oil (conditioned Kd/K₀) |
| Magenta dotted | Default gas |
| Cyan solid | Simm gas |

For clean and cemented sands the methods are identical (they share the same raw
Kd/K₀). Differences appear only in the silty-sand facies where conditioning is applied.

In [ ]:
def fig_facies_comparison(w):
    """Per-facies Vp depth profiles: default vs Simm conditioning."""
    facies_list = ['Clean Sand','Cemented Sand','Silty Sand 1','Silty Sand 2','Silty Shale']
    fig, axes   = plt.subplots(len(facies_list), 1, figsize=(12, 15), sharex=True)
    depth = w['depth']
    for i, fac in enumerate(facies_list):
        ax   = axes[i]
        mask = w['facies'] == fac
        if not mask.any():
            ax.text(0.5, 0.5, f'No data for {fac}', ha='center', va='center',
                    transform=ax.transAxes)
            continue
        _shade_facies(ax, w[mask])
        ax.plot(w.loc[mask,'Vp'],             depth[mask], 'b-',  lw=1.5, label='Brine')
        ax.plot(w.loc[mask,'Vp_default_oil'], depth[mask], 'r--', lw=1.2, label='Default Oil')
        ax.plot(w.loc[mask,'Vp_oil'],         depth[mask], 'g-',  lw=1.2, label='Simm Oil')
        ax.plot(w.loc[mask,'Vp_default_gas'], depth[mask], 'm:',  lw=1.2, label='Default Gas')
        ax.plot(w.loc[mask,'Vp_gas'],         depth[mask], 'c-',  lw=1.2, label='Simm Gas')
        ax.set_ylim(depth[mask].max()+2, depth[mask].min()-2)
        ax.set_ylabel('Depth (m)', fontsize=9)
        ax.set_title(f'{fac}', fontsize=11, fontweight='bold')
        ax.grid(axis='x', alpha=0.3); ax.legend(fontsize=8, loc='lower right')
    axes[-1].set_xlabel('Vp (km/s)', fontsize=10)
    fig.suptitle('Facies-Specific Fluid Substitution: Default (Raw Kd) vs Simm (Conditioned)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    return fig

fig5 = fig_facies_comparison(well)
plt.show()

## 15. Figure 6 — Quantifying the Impact of Dry-Rock Conditioning

Four difference crossplots (Default minus Simm) to show **where and how much**
the adaptive dry-rock trend changes the predicted fluid substitution:

| Panel | Quantity |
|-------|----------|
| (a) | ΔVp_oil vs porosity |
| (b) | ΔVp_gas vs porosity |
| (c) | ΔVp_oil vs Vsh |
| (d) | ΔPR_oil vs porosity |

Points above zero mean the default method predicts **higher** values than Simm's
conditioned approach. Silty sands (where conditioning is applied) will show the
largest deviations; clean sands and shales cluster near zero.

In [ ]:
def fig_default_vs_simm_differences(w):
    """Difference crossplots: (Default − Simm) for oil and gas substitutions."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    ax1, ax2, ax3, ax4 = axes.flatten()
    labelled = w[w['facies'] != 'Background']

    for ax, xcol, ycol, title in [
        (ax1, 'phi', 'Vp_default_oil - Vp_oil',  '(a)  Oil ΔVp vs Porosity'),
        (ax2, 'phi', 'Vp_default_gas - Vp_gas',  '(b)  Gas ΔVp vs Porosity'),
        (ax3, 'Vsh', 'Vp_default_oil - Vp_oil',  '(c)  Oil ΔVp vs Shale Volume'),
        (ax4, 'phi', 'PR_default_oil - PR_oil',   '(d)  Oil ΔPR vs Porosity'),
    ]:
        for fac, grp in labelled.groupby('facies'):
            # evaluate the difference expression
            yvals = grp.eval(ycol)
            ax.scatter(grp[xcol], yvals, c=FACIES_COLORS[fac],
                       s=15, alpha=0.7, edgecolors='none', label=fac)
        ax.axhline(0, color='black', ls='--', alpha=0.5)
        xlabel = 'Effective Porosity φ_e' if xcol == 'phi' else 'Vsh  (v/v)'
        ylabel = ycol.replace(' - ', ' − ').replace('_', ' ')
        ax.set_xlabel(xlabel, fontsize=10)
        ax.set_ylabel(f'Δ  ({ylabel.split("−")[0].strip()})  (Default − Simm)', fontsize=9)
        ax.set_title(title, fontsize=11)
        ax.legend(fontsize=7, markerscale=1.5, ncol=2); ax.grid(True, alpha=0.25)

    fig.suptitle(
        'Impact of Dry Rock Conditioning: Default vs Simm (Positive Δ = Default higher)',
        fontsize=13, fontweight='bold')
    plt.tight_layout()
    return fig

fig6 = fig_default_vs_simm_differences(well)
plt.show()